# 04 — Variáveis derivadas e ABT
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:**
1. **Testa as variáveis derivadas**, cada uma contra a variável original de onde vem (seção 1).
2. **Monta a ABT** (*Analytical Base Table*, tabela analítica de base), a tabela pronta para modelar, a partir da silver: todas as variáveis
   candidatas, limpas, com o mesmo formato no treino e na inferência (seção 2).
3. **Faz a triagem das variáveis e grava a camada gold**, com um dicionário que traz, para cada coluna, o IV
   (*Information Value*, o poder de separar quem sai de quem fica) e a **sugestão da triagem** (recomendada, candidata
   ou sugiro descartar), com o motivo (seção 3).

**Divisão de papéis (D18):** este notebook entrega a ABT e faz a **triagem**; a combinação de variáveis é escolhida no
05 (seleção de variáveis) e a final no 05 (seções 2 e 3). Por isso a ABT traz todas as candidatas, inclusive as que eu
sugiro descartar.

**Três etapas, três nomes:**

| Etapa | Onde | O que faz | Como olha as variáveis |
|---|---|---|---|
| **Triagem** | **04 (este notebook)** | Dá uma nota a cada coluna: recomendada, candidata ou sugiro descartar, pelo IV e pela repetição | Uma de cada vez |
| **Seleção final** | 05, seções 2 e 3 | Testes estatísticos com correção para múltiplos testes e retirada das duplicatas exatas: 41 colunas | Uma de cada vez e em pares |
| **Conferência** | 05, seções 4 e 5 | Seleção para frente por AUC, *stepwise* e VIF, num modelo | Todas juntas, num modelo |
| **Modelagem** | 06 | Regularização (Lasso, Ridge, Elastic Net) e 6 algoritmos sobre as 41 colunas | O modelo que vai para a entrega |


| Tabela gold | Conteúdo |
|---|---|
| `gold_churn_treino` | 5.634 clientes: `customerID`, todas as variáveis candidatas e `Churn` (1 = saiu) |
| `gold_churn_inferencia` | 1.409 clientes: as mesmas colunas, na mesma ordem, sem `Churn` |
| `gold_churn_dicionario` | Uma linha por coluna da ABT: origem, tipo de derivada, fórmula, IV e sugestão |
| `gold_churn_texto` | `customerID` e o `CustomerFeedback`, **só para o modelo com texto** (contraexemplo, D9) |

## Checklist de decisões

**A mesma tabela em todos os notebooks.** ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## Glossário de siglas

Toda sigla usada nos notebooks, por extenso, com o lugar onde ela é explicada em detalhe.

| Sigla | Por extenso | O que é, em uma frase | Onde entender melhor |
|---|---|---|---|
| **IV** | *Information Value* (valor da informação) | Quanto uma variável, sozinha, separa quem sai de quem fica (0 = nada; acima de 0,5 = muito forte) | 03: seção 7.1 e Apêndice A.7 · 04: seção 1 (exemplo calculado) |
| **WoE** | *Weight of Evidence* (peso da evidência) | Em cada grupo, ln(% dos que saíram ÷ % dos que ficaram); positivo = risco acima da média. É a base do IV | 03: Apêndice A.7 · 04: seção 1 |
| **PSI** | *Population Stability Index* (índice de estabilidade da população) | Quanto a distribuição de uma variável mudou entre o histórico e a inferência (abaixo de 0,10 = estável) | 03: seção 5.9 (com exemplo e cenário inventado) |
| **ABT** | *Analytical Base Table* (tabela analítica de base) | A tabela que o modelo lê: uma linha por cliente, uma coluna por variável, tudo limpo | 04: seção 2 |
| **KS** | Teste de Kolmogorov-Smirnov | Testa se duas distribuições de números são iguais | 03: seção 5.9 |
| **CV** | Coeficiente de variação | Desvio-padrão ÷ média, em %: quanto os valores variam em relação ao tamanho deles | 03: seção 5.1 |
| **Q1, Q3, IQR** | Primeiro e terceiro quartis; *interquartile range* (amplitude interquartil) | Q1 e Q3 deixam 25% dos valores abaixo e acima; IQR = Q3 − Q1, a largura da caixa do boxplot | 03: seção 5.3 (critério de Tukey) |
| **V** | V de Cramér | Associação entre duas variáveis de categorias, de 0 a 1 | 03: seção 6 e Apêndice A.1 · nota "Associação entre variáveis" |
| **ρ** (rô) | Correlação de Spearman | Associação entre duas variáveis numéricas, pela ordem dos valores | 03: Apêndice A.2 |
| **η** (eta) | Razão de correlação | Quanto uma variável de categorias explica de uma numérica | 03: Apêndice A.3 |
| **r** | Correlação rank-bisserial | Tamanho do efeito do teste de Mann-Whitney (quem saiu × quem ficou) | 03: Apêndice A.6 |
| **p** | p-valor | A chance de ver uma diferença desse tamanho só por acaso; abaixo de 0,05, dificilmente é acaso | 03: Apêndice A.1 |
| **D1 a D18** | Decisões do checklist | Cada decisão do grupo, com o status (☑ decidido, ◐ sugestão, ☐ em aberto) | Tabela "Checklist de decisões" no início de cada notebook |
| **bronze, silver, gold** | Camadas da arquitetura medalhão | Dado bruto (bronze), limpo (silver) e pronto para o modelo (gold) | 01: seção 2 · 02 e 04 |

In [ ]:
%run ./_comum

## Fonte da seção 1: tabelas bronze

A seção 1 repete o teste das derivadas sobre o dado bruto, como o 03 (os inválidos ficam de fora do cálculo). A ABT, na seção 2, parte da silver.

In [ ]:
hist_raw = spark.table(f"{FQ}.bronze_history").toPandas().drop(columns=["_origem", "_carregado_em"])
inf_raw = spark.table(f"{FQ}.bronze_inference").toPandas().drop(columns=["_origem", "_carregado_em"])
assert (len(hist_raw), len(inf_raw)) == (5634, 1409), "a bronze não tem o número de clientes esperado: rodar o notebook 01"
print(f"bronze_history: {virg(len(hist_raw), 0)} clientes · bronze_inference: {virg(len(inf_raw), 0)} clientes")

## 1. Variáveis derivadas

### O que é uma variável derivada

Uma **variável derivada** é uma coluna nova, calculada a partir das originais. Ela não traz dado novo: **reorganiza** a
informação que já existe para que o modelo (e as pessoas) a enxerguem melhor. É o que se chama de *feature engineering*.

**Por que criar, se a informação já está nas originais?** Porque o modelo não "vê" tudo sozinho:
- uma **regressão logística** soma efeitos, um para cada variável. Se o risco depende de **duas coisas ao mesmo tempo**
  (contrato mensal **e** cheque eletrônico), ela só capta isso se existir uma coluna para a combinação;
- uma relação que **não é uma reta** (o churn despenca nos primeiros meses e depois quase não muda) fica mais fácil de
  capturar com faixas ou com um indicador "cliente novo";
- para o **negócio**, "cliente nos primeiros 6 meses" é mais fácil de usar numa campanha do que "tenure = 4".

**Os tipos de derivada que testamos, com exemplos dos nossos dados:**

| Tipo | O que faz | Exemplo | Quando ajuda |
|---|---|---|---|
| **Indicador** | Vira 1/0 uma condição | `is_monthly_contract`: contrato mensal = 1 | Resume uma variável com várias categorias no que importa |
| **Interação** | Junta duas condições com "e" | `IsHighRiskContract`: mensal **e** cheque eletrônico | O risco da combinação é maior que a soma das partes |
| **Faixas** | Corta um número em intervalos | `tenure_group`: 0-6, 7-12, 13-24... meses | Relação que não é uma reta; leitura para o negócio |
| **Razão** | Divide uma variável por outra | `mensalidade_relativa_pacote`: preço ÷ média do pacote | Compara o cliente com quem é parecido com ele |
| **Diferença** | Subtrai uma da outra | `charge_difference`: mensalidade atual − média histórica | Mostra mudança ao longo do tempo |
| **Contagem** | Soma vários sim/não | `n_protecao`: quantos dos 4 serviços de proteção tem | Resume várias colunas parecidas numa só |
| **Mudança de escala** | Muda a unidade | `tenure_years`: meses ÷ 12 | Só leitura: não acrescenta informação |

### Como avaliamos cada derivada: o IV (*Information Value*, valor da informação)

O primeiro filtro é o mesmo da seção 7.1 do 03: o **IV** (*Information Value*; teoria completa no Apêndice A.7 do 03), que mede o quanto uma variável, **sozinha**, separa quem
sai de quem fica.

> **Como o IV é calculado, passo a passo, com `is_monthly_contract`**
>
> 1. Para cada grupo, calculo **que fração dos que saíram** e **que fração dos que ficaram** está nele:
>
> | Grupo | Saíram | % dos que saíram | Ficaram | % dos que ficaram |
> |---|---|---|---|---|
> | Mensal (1) | 1.322 | 87,9% | 1.777 | 43,0% |
> | Não mensal (0) | 182 | 12,1% | 2.353 | 57,0% |
>
> 2. O **WoE** (*Weight of Evidence*, peso da evidência) de cada grupo é o logaritmo da razão entre as duas frações:
>    WoE = ln(% dos que saíram ÷ % dos que ficaram).
>    - Mensal: ln(87,9 ÷ 43,0) = **+0,71**: o grupo tem **mais** gente que sai do que gente que fica (risco acima da média).
>    - Não mensal: ln(12,1 ÷ 57,0) = **−1,55**: o grupo é dominado por quem fica (risco abaixo da média).
> 3. Cada grupo contribui com (% saíram − % ficaram) × WoE. As duas partes têm sempre o mesmo sinal, então a
>    contribuição é sempre positiva:
>    - Mensal: (0,879 − 0,430) × 0,71 = 0,32
>    - Não mensal: (0,121 − 0,570) × (−1,55) = 0,70
> 4. O **IV é a soma: 1,02**. Muito forte.
>
> **Leitura:** IV = 0 quando os dois grupos têm a mesma proporção de quem sai e de quem fica (a variável não separa nada).
> Quanto mais as proporções se afastam, maior o IV.
>
> | IV | Poder |
> |---|---|
> | abaixo de 0,02 | sem poder |
> | 0,02 a 0,1 | fraco |
> | 0,1 a 0,3 | médio |
> | 0,3 a 0,5 | forte |
> | acima de 0,5 | muito forte (conferir se não é vazamento) |
>
> **Variáveis numéricas** (como `ChargesRatio`) são divididas em **10 faixas com o mesmo número de clientes**
> (decis), e cada faixa é um grupo.

### Por que comparo cada derivada com a original de onde vem

Uma derivada feita de **uma só** variável (indicador, faixas, mudança de escala) **não pode ter mais informação que a
original**: ela só agrupa ou renomeia os valores. Por isso o IV dela fica **igual ou abaixo** do IV da original. O que
ela pode ganhar é **simplicidade**: `is_monthly_contract` guarda 1,02 dos 1,17 de `Contract` com uma única coluna 1/0.

Já uma derivada feita de **duas variáveis** (interação, razão, diferença) pode trazer algo que nenhuma das duas mostra
sozinha. Mas o IV olha **uma variável por vez**: mesmo que a interação tenha IV menor que a original, ela pode
acrescentar informação **junto** com as originais no modelo. Quem responde isso é a **seleção de variáveis**, feita no
modelo.

**Resumo do critério:** o IV é o primeiro filtro (separa ou não separa); a comparação com a original diz se a derivada
**repete** a original ou **acrescenta** algo; a decisão final é no modelo.

In [ ]:
# alvo e funções de IV (as mesmas da seção 7.1 do notebook 03)
y = hist_raw["Churn"].eq("Yes").astype(int)

def iv_woe(grupos: pd.Series, y: pd.Series) -> float:
    t = pd.crosstab(grupos, y)
    p1 = (t[1] / t[1].sum()).clip(lower=1e-4)
    p0 = (t[0] / t[0].sum()).clip(lower=1e-4)
    return float(((p1 - p0) * np.log(p1 / p0)).sum())

def faixa_iv(iv):
    return ("sem poder" if iv < 0.02 else "fraco" if iv < 0.10 else "médio" if iv < 0.30
            else "forte" if iv < 0.50 else "muito forte (conferir vazamento)")

In [ ]:
# 1. Teste das variáveis derivadas: o IV de cada uma contra a variável original de onde vem
num = lambda c: pd.to_numeric(hist_raw[c], errors="coerce")       # 'unknown' e 'null' viram NaN (e não −999)
tn8, mc8, tc8, renda8 = num("tenure"), num("MonthlyCharges"), num("TotalCharges"), num("MonthlyIncome")
esperado = tn8 * mc8
media_mensal = tc8 / tn8.replace(0, np.nan)
sim = lambda c: hist_raw[c].str.lower().eq("yes")
mensal = hist_raw["Contract"].eq("Month-to-month")
servicos_j = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
              "PhoneService"]
adicionais = servicos_j[:-1]

# nome: (tipo de derivada, valores, fórmula, variável original de referência, contínua?)
DERIVADAS = {
    "IncomeCommitment": ("razão", mc8 / renda8, "MonthlyCharges ÷ MonthlyIncome", "MonthlyCharges", True),
    "ChargesRatio": ("razão", tc8 / esperado.where(esperado > 0), "TotalCharges ÷ (tenure × MonthlyCharges)", "TotalCharges", True),
    "TotalServicesSubscribed": ("contagem", sum(sim(c).astype(int) for c in servicos_j), "nº de serviços com Yes, com telefonia", "OnlineSecurity", False),
    "HasFiberOpticWithoutTechSupport": ("interação", (hist_raw["InternetService"].eq("Fiber optic") & hist_raw["TechSupport"].eq("No")).astype(int),
                                        "fibra ótica e sem suporte técnico", "TechSupport", False),
    "IsHighRiskContract": ("interação", (mensal & hist_raw["PaymentMethod"].eq("Electronic check")).astype(int),
                           "contrato mensal e cheque eletrônico", "Contract", False),
    "TenureCohort": ("faixas", pd.cut(tn8, [-1, 5, 12, 24, np.inf], labels=["0-5", "6-12", "13-24", "25+"]), "4 faixas de tenure", "tenure", False),
    "tenure_years": ("mudança de escala", tn8 / 12, "tenure ÷ 12", "tenure", True),
    "is_new_customer": ("indicador", (tn8 <= 6).astype(float).where(tn8.notna()), "tenure até 6 meses", "tenure", False),
    "tenure_group": ("faixas", pd.cut(tn8, [-1, 6, 12, 24, 48, np.inf], labels=["0-6", "7-12", "13-24", "25-48", "49+"]), "5 faixas de tenure", "tenure", False),
    "income_after_charges": ("diferença", renda8 - mc8, "MonthlyIncome − MonthlyCharges", "MonthlyIncome", True),
    "average_monthly_charges": ("razão", media_mensal, "TotalCharges ÷ tenure", "MonthlyCharges", True),
    "charge_difference": ("diferença", mc8 - media_mensal, "MonthlyCharges − TotalCharges ÷ tenure", "MonthlyCharges", True),
    "total_addons": ("contagem", sum(sim(c).astype(int) for c in adicionais), "nº de serviços adicionais, sem telefonia", "OnlineSecurity", False),
    "has_family": ("indicador", (sim("Partner") | sim("Dependents")).astype(int), "tem parceiro(a) ou dependentes", "Dependents", False),
    "no_support_with_internet": ("interação", (hist_raw["InternetService"].ne("No") & hist_raw["TechSupport"].eq("No")).astype(int),
                                 "tem internet e não tem suporte", "TechSupport", False),
    "is_monthly_contract": ("indicador", mensal.astype(int), "contrato mensal", "Contract", False),
    "is_electronic_check": ("indicador", hist_raw["PaymentMethod"].eq("Electronic check").astype(int), "paga por cheque eletrônico", "PaymentMethod", False),
    "new_monthly_customer": ("interação", ((tn8 <= 6) & mensal).astype(float).where(tn8.notna()), "até 6 meses e contrato mensal", "Contract", False),
}
iv_orig = spark.table(f"{FQ}.dicionario_dados").toPandas().set_index("coluna")["IV"].astype(float)   # IV das originais (notebook 03)
linhas_d = []
for nome, (tipo, x, formula, ref, continua) in DERIVADAS.items():
    ok = x.notna()
    grupos = pd.qcut(x[ok], 10, duplicates="drop") if continua else x[ok]   # contínua: decis, como na 7.1
    taxa = y[ok].groupby(grupos, observed=True).mean()
    iv_d = iv_woe(grupos, y[ok])
    linhas_d.append({"derivada": nome, "tipo": tipo, "formula": formula, "n_validos": int(ok.sum()), "IV": iv_d,
                     "poder_iv": faixa_iv(iv_d), "churn_min_pct": 100 * taxa.min(), "churn_max_pct": 100 * taxa.max(),
                     "original_referencia": ref, "IV_original": iv_orig[ref]})
deriv = pd.DataFrame(linhas_d).sort_values("IV", ascending=False).reset_index(drop=True)
deriv["IV_derivada_menos_original"] = deriv["IV"] - deriv["IV_original"]
display(deriv.round(3))

# conferências das afirmações dos notebooks
ic = DERIVADAS["IncomeCommitment"][1]
print(f"Churn com a conta acima de 1% da renda: {virg(100 * y[ic > 0.01].mean(), 1)}%; "
      f"até 1%: {virg(100 * y[ic <= 0.01].mean(), 1)}%")
fibra = hist_raw["InternetService"].eq("Fiber optic")
print(f"Clientes de fibra: churn de {virg(100 * y[fibra & hist_raw['TechSupport'].eq('No')].mean(), 1)}% sem suporte "
      f"e {virg(100 * y[fibra & hist_raw['TechSupport'].eq('Yes')].mean(), 1)}% com suporte")
from scipy.stats import spearmanr
print(f"Gasto médio mensal × mensalidade atual: Spearman {virg(spearmanr(media_mensal, mc8, nan_policy='omit')[0], 2)}")
print("Churn (%) pelo número de serviços (com telefonia) e de adicionais (sem telefonia):")
display(pd.DataFrame({"TotalServicesSubscribed": 100 * y.groupby(DERIVADAS["TotalServicesSubscribed"][1]).mean(),
                      "total_addons": 100 * y.groupby(DERIVADAS["total_addons"][1]).mean()})
        .round(1).rename_axis("n_servicos").reset_index())

In [ ]:
# 1b As derivadas ao lado da variável original de onde vêm
d82 = deriv.sort_values("IV")
fig, ax = plt.subplots(figsize=(12, 10))
yy = np.arange(len(d82))
for lim, rot in [(0.10, "fraco | médio"), (0.30, "médio | forte"), (0.50, "forte | muito forte")]:
    ax.axvline(lim, color="#cccccc", linestyle=":", linewidth=1, zorder=0)
    ax.text(lim, len(d82) - 0.3, rot, ha="center", fontsize=8, color="#888888")
ax.barh(yy, d82["IV"], height=0.6, color=[COR if v >= 0.5 else CINZA for v in d82["IV"]], zorder=2)
ax.scatter(d82["IV_original"], yy, marker="D", s=45, color=DESTAQUE, zorder=3)
for y0, (_, r) in zip(yy, d82.iterrows()):
    dentro = r["IV"] > 0.2                                   # rótulo dentro da barra, para não cobrir o losango
    longe = r["IV_original"] - r["IV"] > 0.12                  # losango longe: rótulo logo depois da barra
    x_rot = r["IV"] - 0.012 if dentro else r["IV"] + 0.012 if longe else max(r["IV"], r["IV_original"]) + 0.03
    ax.text(x_rot, y0 if dentro or longe else y0 - 0.24, f"IV {virg(r['IV'], 2)}", va="center",
            ha="right" if dentro else "left", fontsize=8.5, color="white" if dentro else TEXTO, fontweight="bold")
    ax.text(r["IV_original"] + 0.025, y0 + 0.2, f"{r['original_referencia']} {virg(r['IV_original'], 2)}",
            va="center", fontsize=8, color=DESTAQUE)
ax.set_yticks(yy, [f"{n} [{a}]\n({f})" for n, a, f in zip(d82["derivada"], d82["tipo"], d82["formula"])], fontsize=8)
ax.set_xlim(0, 1.45)
ax.set_ylim(-0.6, len(d82) - 0.1)
ax.set_xlabel("IV (Information Value): barra = derivada, losango laranja = variável original de referência")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
fig.suptitle("Nenhuma das 18 derivadas supera, sozinha, a variável original de onde vem;\n"
             "as melhores são indicadores simples: contrato mensal, faixas de tempo de casa e internet sem suporte",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "tenure", "TechSupport", "InternetService", "PaymentMethod", "OnlineSecurity",
             "MonthlyCharges", "TotalCharges", "MonthlyIncome", "Dependents", "Partner"])
plt.show()

> **Como ler a tabela 1**
>
> | Coluna | O que é |
> |---|---|
> | `derivada`, `tipo`, `formula` | O nome, o tipo (indicador, interação, faixas...) e como é calculada |
> | `n_validos` | Clientes com valor. É menor que 5.634 quando a derivada usa `tenure` (19 inválidos) ou `TotalCharges` (278 inválidos): aqui o dado é o bruto, sem preenchimento |
> | `IV`, `poder_iv` (IV = *Information Value*) | O poder de separar quem sai de quem fica, e a faixa (sem poder a muito forte) |
> | `churn_min_pct`, `churn_max_pct` | A menor e a maior taxa de churn entre os grupos da derivada. Quanto mais longe, mais ela separa |
> | `original_referencia`, `IV_original` | A variável original de onde a derivada vem (nas combinações, a mais forte delas) e o IV dela |
> | `IV_derivada_menos_original` | Negativo = a derivada separa **menos** que a original sozinha; zero = separa igual |
>
> **As linhas impressas abaixo da tabela** conferem quatro afirmações:
> - **Conta acima de 1% da renda: 31,0% de churn; até 1%: 19,3%.** Parece forte, mas a renda não tem poder (IV 0,004)
>   e parece gerada (seções 5 e 6 do 03): a diferença vem da **mensalidade** no numerador, e não da renda.
> - **Fibra: 49,3% de churn sem suporte técnico e 22,9% com suporte.** Entre os clientes de fibra, não ter suporte
>   **dobra** o risco: é a interação `HasFiberOpticWithoutTechSupport`.
> - **Gasto médio mensal × mensalidade atual: Spearman 0,99.** `average_monthly_charges` é praticamente a própria
>   mensalidade: repete `MonthlyCharges`.
> - **Churn pelo número de serviços:** a tabela mostra que o churn **não cai em linha reta** com o número de serviços,
>   porque o "0 serviços" mistura quem não contratou nada com quem **não tem internet** (7% de churn). Por isso as
>   contagens gerais perdem para `n_protecao` (seção 2).
>
> **Como ler o gráfico 1b**
> - **A barra é o IV da derivada**; azul = 0,50 ou mais, cinza = abaixo.
> - **O losango laranja é o IV da variável original** de referência.
> - **O que procurar:** se a barra passa do losango, a derivada sozinha separa melhor que a original. **Nenhuma passa**;
>   `tenure_years` empata, porque é só `tenure` em outra unidade. Isso confirma a teoria acima: as derivadas valem pela
>   **simplicidade** (indicadores) ou pelo que acrescentam **junto** com as originais (interações), e não pelo IV sozinho.

> **Explicação (1): o que cada derivada acrescenta**
>
> **Teoria aplicada:** uma variável derivada vale a pena quando separa quem sai de quem fica melhor do que as
> originais, ou quando traz uma combinação que as originais não mostram sozinhas. O primeiro filtro é o IV, o mesmo da 7.1.
> Foram testadas 18 derivadas.
>
> | Derivada | Tipo | IV | Original | Leitura | Sugestão |
> |---|---|---|---|---|---|
> | `is_monthly_contract` | indicador | 1,02 | `Contract` 1,17 | 42,7% de churn no mensal contra 7,2% nos demais | Candidata: resume o contrato em uma variável 1/0 |
> | `tenure_years` | mudança de escala | 0,77 | `tenure` 0,77 | É `tenure` dividido por 12 | Não usar: repete `tenure` |
> | `tenure_group` | faixas | 0,68 | `tenure` 0,77 | 52,4% até 6 meses e 9,5% acima de 48 | Para a apresentação; no modelo, `tenure` contínua guarda mais informação |
> | `no_support_with_internet` | interação | 0,64 | `TechSupport` 0,71 | 41,8% contra 11,8% | Candidata; versão mais ampla da de fibra |
> | `IsHighRiskContract` | interação | 0,61 | `Contract` 1,17 | 53,6% contra 17,3% | Candidata: interação útil para modelo linear |
> | `HasFiberOpticWithoutTechSupport` | interação | 0,60 | `TechSupport` 0,71 | 49,3% contra 16,1%; entre os de fibra, sem suporte o churn é 2,2 vezes o com suporte (49,3% contra 22,9%) | Candidata: a afirmação "dobra" se confirma |
> | `TenureCohort` | faixas | 0,59 | `tenure` 0,77 | 54,2% nos primeiros 6 meses contra 14,6% acima de 2 anos | Mesmo papel de `tenure_group`; ficar com uma só |
> | `new_monthly_customer` | interação | 0,46 | `Contract` 1,17 | 54,7% contra 19,7% | Candidata: interação contrato mensal × cliente novo, que já estava na `potencial_derivada` |
> | `is_electronic_check` | indicador | 0,43 | `PaymentMethod` 0,44 | 45,1% contra 17,4% | Candidata: guarda quase todo o IV da forma de pagamento em 1/0 |
> | `is_new_customer` | indicador | 0,41 | `tenure` 0,77 | 52,4% contra 19,8% | É a `cliente_novo` da 8.1 |
> | `average_monthly_charges` | razão | 0,34 | `MonthlyCharges` 0,38 | Spearman de 0,99 com a mensalidade atual | Não usar: repete `MonthlyCharges` |
> | `charge_difference` | diferença | 0,33 | `MonthlyCharges` 0,38 | Mediana 0: mesma ideia da `ChargesRatio` | Testar uma das duas no 05, não as duas |
> | `total_addons` | contagem | 0,30 | `OnlineSecurity` 0,72 | O churn não cai em linha: 20,7% com 0 e 45,2% com 1. O 0 junta quem não tem internet | Separar em `n_protecao` e `n_streaming` |
> | `TotalServicesSubscribed` | contagem | 0,27 | `OnlineSecurity` 0,72 | Mesmo problema: 45,6% com 0, 20,8% com 1, 42,9% com 2 | Separar em `n_protecao` e `n_streaming` |
> | `ChargesRatio` | razão | 0,18 | `TotalCharges` 0,31 | Mediana 1,00: é a regra de ±20% da 4.2 virada variável | Testar uma das duas no 05 |
> | `IncomeCommitment` | razão | 0,16 | `MonthlyCharges` 0,38 | Dividir a mensalidade por uma renda sem poder (IV 0,004) só dilui a informação | Não usar |
> | `has_family` | indicador | 0,13 | `Dependents` 0,16 | 34,1% sem família contra 20,3% | É o oposto da `sem_familia` da 8.1; não supera `Dependents` |
> | `income_after_charges` | diferença | 0,005 | `MonthlyIncome` 0,004 | Renda menos uma mensalidade pequena é quase a própria renda | Não usar: sem poder |
>
> **Para o negócio:** as derivadas que funcionam são indicadores simples, fáceis de explicar ao time de vendas:
> contrato mensal, cliente nos primeiros 6 meses, internet sem suporte técnico e pagamento por cheque eletrônico. As
> combinações mostram onde o risco se concentra: "contrato mensal pagando por cheque eletrônico" (53,6% saem) e
> "cliente novo no contrato mensal" (54,7% saem).

> **Triagem das derivadas (a combinação é sugerida no 05 e decidida na modelagem, D18)**
>
> 1. Candidatas: `is_monthly_contract`, `no_support_with_internet`, `IsHighRiskContract`, `HasFiberOpticWithoutTechSupport`, `new_monthly_customer`, `is_electronic_check`, `is_new_customer`.
> 2. Sugiro descartar: `tenure_years`, `average_monthly_charges`, `IncomeCommitment`, `income_after_charges`, `total_addons`, `TotalServicesSubscribed` (trocadas por `n_protecao` e `n_streaming`).
> 3. Todas estão na ABT, com a sugestão no dicionário (`gold_churn_dicionario`).
>
> **Respostas e decisões:**
>
> - **D19:** a conferência do notebook 05 (seção 5) escolhe as 9 recomendadas + `new_monthly_customer` + `is_electronic_check` (AUC 0,841). A seleção final é feita no 05 (seções 2 e 3) e conferida com os métodos da aula.

## 2. A ABT (*Analytical Base Table*): tabela pronta para modelar

**O que faço:** parto da silver (limpa e tipada, notebook 02) e acrescento as variáveis derivadas. O resultado é a ABT:
uma linha por cliente, todas as variáveis candidatas, sem valores ausentes, com as **mesmas colunas no treino e na inferência**.

**O que entra:**
| Grupo | Colunas | Por quê |
|---|---|---|
| Chave | `customerID` | Não é variável do modelo; identifica o cliente no `Prediction.csv` (D13) |
| Originais limpas | As 20 explicativas da silver | Inclusive as que sugiro descartar (D12, D15, D16): a combinação é sugerida no 05 e decidida na modelagem |
| Serviços nas duas versões | Originais (3 categorias) e `_bin` (tem = 1) | D10: eles escolhem a versão. A `_bin` deve ir junto com `InternetService` |
| Indicador de ausência | `fl_total_charges_ausente` | D5: testar com e sem |
| Derivadas | As 18 da seção 1, mais `n_protecao`, `n_streaming`, `pagamento_automatico` e `mensalidade_relativa_pacote` | Para a triagem e a seleção (05) |
| Alvo | `Churn` (1/0) | Só no treino |

**O que não entra:** o `CustomerFeedback` (vazamento, D9). Ele vai na `gold_churn_texto`, com o indicador
`fl_menciona_cancelamento`, só para o modelo com texto.

**Uma regra importante:** tudo o que é **aprendido dos dados** vem só do treino e é aplicado igual na inferência. Aqui,
é a mensalidade média de cada tipo de internet (para a `mensalidade_relativa_pacote`).

**Ajustes nas derivadas por causa da limpeza:** com a silver, não há mais valores ausentes. Nos 10 + 1 clientes com
`tenure` = 0, as razões que dividem por `tenure` ficariam indefinidas; nelas uso a própria mensalidade
(`average_monthly_charges` = `MonthlyCharges`, `charge_difference` = 0, `ChargesRatio` = 1).

### Teoria: o que é uma ABT e as regras que ela precisa seguir

**ABT** (*Analytical Base Table*, tabela analítica de base) é a tabela que o modelo lê: **uma linha por cliente**, **uma coluna por variável**,
tudo já limpo. É o produto da camada gold. O modelo não sabe de onde os dados vieram; ele só confia que a tabela segue
algumas regras. Se uma regra falha, o modelo erra sem avisar.

| Regra | Por quê | Como conferimos (2.2) |
|---|---|---|
| **Uma linha por cliente** | Cada cliente conta uma vez no treino e recebe uma previsão | `customerID` único |
| **Mesmas colunas, na mesma ordem, no treino e na inferência** | O modelo treinado espera exatamente as colunas que viu | Comparação das listas de colunas |
| **Sem valores ausentes** | A maioria dos modelos não aceita vazio, e cada um trataria de um jeito | Contagem de ausentes = 0 |
| **Nada que só existe depois do desfecho** | É o vazamento: o modelo aprenderia o que não terá na hora de prever | O texto fica fora (D9) |
| **Tudo o que é aprendido vem só do treino** | Senão, a inferência "ensina" o modelo, e a avaliação fica otimista | A média por pacote vem só do histórico |
| **Nenhum cliente nas duas bases** | Um cliente da inferência visto no treino infla o acerto | Interseção de `customerID` vazia |

> **Vazamento na preparação: um exemplo**
> A `mensalidade_relativa_pacote` divide a mensalidade pela **média do pacote**. Se essa média fosse calculada com o
> histórico **e** a inferência juntos, os clientes que o modelo vai pontuar entrariam no cálculo de uma variável do
> treino. Parece inofensivo, mas é a mesma lógica do vazamento: usar informação que não existiria no momento de treinar.
> Por isso a média é calculada **só no histórico** e aplicada igual às duas bases (a mesma regra da mediana do `tenure`, D4).

### Teoria: por que as categorias ficam como texto, e o cuidado com os serviços

**Modelos diferentes pedem codificações diferentes:**
- **Regressão logística** precisa de números: cada categoria vira uma coluna 0/1 (*one-hot* ou *dummies*), menos uma,
  que fica como referência. Se todas entrarem, uma é a soma das outras e o modelo não consegue estimar os coeficientes
  (*armadilha da variável dummy*).
- **Árvores** (árvore de decisão, random forest, gradient boosting) usam as categorias quase direto.

Por isso a ABT entrega as categorias **como texto** (D14): cada modelo codifica do seu jeito.

**O cuidado com os serviços (D10).** Na versão original, cada serviço tem a categoria "No internet service". Ela é
**exatamente** o mesmo grupo de `InternetService` = No. Numa logística com `InternetService` e os 6 serviços
originais, a dummy "No internet service" aparece **7 vezes**, idêntica: é **colinearidade perfeita**, e o modelo não
consegue separar o efeito de cada uma. Com os serviços `_bin` (tem = 1) **junto com** `InternetService`, cada
informação aparece uma vez só: "não tem internet" fica em `InternetService`, e o `_bin` diz se quem tem internet
contratou o serviço.

In [ ]:
# 2.1 Lê a silver e define as derivadas (as mesmas fórmulas da seção 1, agora sobre o dado limpo)
silver_hist = spark.table(f"{FQ}.silver_history").toPandas()
silver_inf = spark.table(f"{FQ}.silver_inference").toPandas()
assert (len(silver_hist), len(silver_inf)) == (5634, 1409), "a silver não tem o número de clientes esperado: rodar o notebook 02"

SERVICOS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
PROTECAO = SERVICOS[:4]
MEDIA_PACOTE = silver_hist.groupby("InternetService")["MonthlyCharges"].mean()   # aprendida só no treino

def montar_abt(df: pd.DataFrame) -> pd.DataFrame:
    d = df.drop(columns=["CustomerFeedback"]).copy()                 # D9: o texto não entra na ABT
    tn, mc, tc, renda = d["tenure"], d["MonthlyCharges"], d["TotalCharges"], d["MonthlyIncome"]
    sim = lambda c: d[c].eq("Yes")
    mensal = d["Contract"].eq("Month-to-month")
    media_mensal = (tc / tn.where(tn > 0)).fillna(mc)                # tenure = 0: a própria mensalidade
    esperado = tn * mc

    for c in SERVICOS:                                               # D10: versão sim/não, ao lado da original
        d[f"{c}_bin"] = sim(c).astype(int)
    novas = {
        # razões, contagens, interações, faixas e indicadores (seção 1)
        "IncomeCommitment": mc / renda,
        "ChargesRatio": (tc / esperado.where(esperado > 0)).fillna(1.0),
        "TotalServicesSubscribed": sum(sim(c).astype(int) for c in SERVICOS + ["PhoneService"]),
        "HasFiberOpticWithoutTechSupport": (d["InternetService"].eq("Fiber optic") & d["TechSupport"].eq("No")).astype(int),
        "IsHighRiskContract": (mensal & d["PaymentMethod"].eq("Electronic check")).astype(int),
        "TenureCohort": pd.cut(tn, [-1, 5, 12, 24, np.inf], labels=["0-5", "6-12", "13-24", "25+"]).astype(str),
        "tenure_years": tn / 12,
        "is_new_customer": (tn <= 6).astype(int),
        "tenure_group": pd.cut(tn, [-1, 6, 12, 24, 48, np.inf], labels=["0-6", "7-12", "13-24", "25-48", "49+"]).astype(str),
        "income_after_charges": renda - mc,
        "average_monthly_charges": media_mensal,
        "charge_difference": mc - media_mensal,
        "total_addons": sum(sim(c).astype(int) for c in SERVICOS),
        "has_family": (sim("Partner") | sim("Dependents")).astype(int),
        "no_support_with_internet": (d["InternetService"].ne("No") & d["TechSupport"].eq("No")).astype(int),
        "is_monthly_contract": mensal.astype(int),
        "is_electronic_check": d["PaymentMethod"].eq("Electronic check").astype(int),
        "new_monthly_customer": ((tn <= 6) & mensal).astype(int),
        # contagens, indicador e razão novos (ideias da seção 8 do 03)
        "n_protecao": sum(sim(c).astype(int) for c in PROTECAO),
        "n_streaming": sim("StreamingTV").astype(int) + sim("StreamingMovies").astype(int),
        "pagamento_automatico": d["PaymentMethod"].str.contains("automatic").astype(int),
        "mensalidade_relativa_pacote": mc / d["InternetService"].map(MEDIA_PACOTE),
    }
    for nome, valores in novas.items():
        d[nome] = valores
    alvo = ["Churn"] if "Churn" in d.columns else []
    return d[[c for c in d.columns if c not in alvo] + alvo]          # Churn sempre na última coluna

abt_treino = montar_abt(silver_hist)
abt_inferencia = montar_abt(silver_inf)
print(f"ABT de treino: {virg(len(abt_treino), 0)} clientes × {abt_treino.shape[1]} colunas")
print(f"ABT de inferência: {virg(len(abt_inferencia), 0)} clientes × {abt_inferencia.shape[1]} colunas (sem Churn)")

In [ ]:
# 2.2 Conferências: o que o modelo precisa poder supor sobre a ABT
assert list(abt_inferencia.columns) == [c for c in abt_treino.columns if c != "Churn"], "colunas diferentes entre treino e inferência"
assert abt_treino.isna().sum().sum() == 0 and abt_inferencia.isna().sum().sum() == 0, "a ABT tem valores ausentes"
assert abt_treino["customerID"].is_unique and abt_inferencia["customerID"].is_unique, "customerID repetido"
assert not set(abt_treino["customerID"]) & set(abt_inferencia["customerID"]), "cliente presente nas duas bases"
assert "CustomerFeedback" not in abt_treino.columns, "o texto não pode entrar na ABT (D9)"
for c in SERVICOS:                                                  # a versão _bin bate com a original
    assert (abt_treino[f"{c}_bin"] == abt_treino[c].eq("Yes")).all()

tipos = abt_treino.drop(columns=["customerID", "Churn"]).dtypes
conf = pd.DataFrame({
    "conferência": ["clientes (treino / inferência)", "colunas candidatas", "  numéricas", "  texto (categorias)",
                    "valores ausentes", "taxa de churn no treino"],
    "resultado": [f"{virg(len(abt_treino), 0)} / {virg(len(abt_inferencia), 0)}",
                  str(len(tipos)), str(int(tipos.map(pd.api.types.is_numeric_dtype).sum())), str(int((~tipos.map(pd.api.types.is_numeric_dtype)).sum())),
                  "0", f"{virg(100 * abt_treino['Churn'].mean(), 1)}%"]})
display(conf)
print("Mensalidade média por tipo de internet (aprendida no treino, usada nas duas bases):")
display(MEDIA_PACOTE.round(2).rename("media_R$").reset_index())

## 3. Triagem: dicionário da ABT com o IV (*Information Value*) e a sugestão de cada coluna

**O que faço:** para cada coluna candidata, calculo o IV **na própria ABT** (com os valores já limpos) e registro a
origem, o tipo de derivada, a fórmula e a **sugestão da triagem**, com o motivo. A triagem olha uma variável por vez;
o 05 parte dela para escolher a combinação.

**Sugestões da triagem:**
- **recomendada:** forte, sem repetir outra mais forte;
- **candidata:** vale testar; costuma ter uma alternativa na ABT (escolher uma das duas);
- **sugiro descartar:** sem poder, repete outra variável ou depende de uma variável sem poder.

**Por que o IV muda um pouco em relação ao 03:** lá, os inválidos ficavam fora do cálculo; aqui, eles já foram
preenchidos (D2 a D4). As diferenças aparecem só em `tenure` e `TotalCharges`, e são pequenas.

### Teoria: o IV de uma variável pode enganar quando outra variável "explica" parte dela

O IV olha **uma variável por vez**. Se parte do poder dela vem de **outra variável**, o IV sozinho engana, para mais
ou para menos. Os serviços mostram os dois lados:

| Serviço | IV na versão original (3 categorias) | IV do `_bin`, todos os clientes | IV do `_bin` só entre quem tem internet |
|---|---|---|---|
| Segurança online | 0,72 | 0,16 | **0,40** |
| Suporte técnico | 0,71 | 0,15 | **0,39** |
| Backup online | 0,53 | 0,03 | 0,17 |
| Proteção de aparelho | 0,50 | 0,02 | 0,13 |
| Streaming de TV | 0,40 | 0,03 | **0,00** |
| Streaming de filmes | 0,40 | 0,02 | **0,01** |

- **Na versão original, o IV é inflado** pela categoria "No internet service" (7% de churn), que é a informação de
  `InternetService`, e não do serviço.
- **No `_bin` com todos os clientes, o IV é achatado**, porque o "não tem" junta quem não contratou (cerca de 40% de
  churn) com quem não tem internet (7%): dois grupos opostos na mesma categoria.
- **O número certo é o do meio do caminho: entre quem tem internet.** É o que o serviço acrescenta **depois** de saber
  o tipo de internet, que é exatamente a situação no modelo (D10: `_bin` junto com `InternetService`).

É o mesmo fenômeno de "sorvete e afogamento" da seção 6 do 03: uma terceira variável (ter ou não internet) cria ou
esconde a associação. Por isso o dicionário traz a coluna `IV_entre_quem_tem_internet` para os serviços.

### Teoria: da triagem à seleção (próximo passo: notebook 05)

A triagem é **univariada** (uma variável por vez). A seleção é **multivariada**: as variáveis entram juntas, e o que
importa é o que cada uma acrescenta às outras. O dicionário prepara essa etapa, feita no 05:
1. **Começar pelas recomendadas.**
2. **Escolher uma de cada grupo de alternativas** (coluna `motivo`): `Contract` **ou** `is_monthly_contract`; os
   serviços originais **ou** os `_bin`; `MonthlyCharges` **ou** `mensalidade_relativa_pacote`. Colocar as duas repete
   informação (multicolinearidade).
3. **Testar as candidatas uma a uma**, vendo se melhoram a métrica de validação, ou usar um método que faz isso
   automaticamente (seleção *stepwise*, que testa variável por variável; regularização L1, também chamada Lasso, que zera os coeficientes das variáveis que pouco ajudam; ou a importância das variáveis numa árvore).
4. **Comparar sempre com a linha de base** da seção 10 do 03: as regras simples, sem modelo.

In [ ]:
# 3.1 Monta o dicionário da ABT
numerica = lambda x: pd.api.types.is_numeric_dtype(x)
def iv_coluna(x: pd.Series, y: pd.Series) -> float:
    continua = numerica(x) and x.nunique() > 10
    grupos = pd.qcut(x, 10, duplicates="drop") if continua else x       # contínua: decis, como na 7.1
    return iv_woe(grupos, y)

ORIGEM_DERIV = {n: (t, f) for n, (t, _, f, _, _) in DERIVADAS.items()}
ORIGEM_DERIV.update({
    **{f"{c}_bin": ("recodificação", f"{c} = Yes → 1; No ou No internet service → 0 (D10)") for c in SERVICOS},
    "n_protecao": ("contagem", "nº de serviços de proteção: segurança, backup, proteção e suporte (0 a 4)"),
    "n_streaming": ("contagem", "nº de streamings: TV e filmes (0 a 2)"),
    "pagamento_automatico": ("indicador", "transferência ou cartão automático = 1"),
    "mensalidade_relativa_pacote": ("razão", "MonthlyCharges ÷ média do mesmo InternetService no treino"),
})

REC, CAND, DESC = "recomendada", "candidata", "sugiro descartar"
SUGESTAO = {
    "Contract": (REC, "A mais forte da base"),
    "tenure": (REC, "Muito forte; o churn despenca com o tempo de casa"),
    "InternetService": (REC, "Muito forte; deve ir junto com os serviços _bin (guarda os 7% de churn de quem não tem internet)"),
    "PaymentMethod": (CAND, "Forte; alternativas: is_electronic_check ou pagamento_automatico"),
    "PaperlessBilling": (REC, "Médio, sem repetição"),
    "Dependents": (REC, "Médio, sem repetição"), "SeniorCitizen": (REC, "Médio, sem repetição"), "Partner": (REC, "Médio, sem repetição"),
    "MonthlyCharges": (CAND, "Forte, mas repete InternetService (η 0,91); alternativa: mensalidade_relativa_pacote (D16)"),
    "TotalCharges": (DESC, "Repete tenure (ρ 0,89): é quase mensalidade × meses (D16)"),
    "fl_total_charges_ausente": (CAND, "Churn de 33,1% × 26,4% (p = 0,016), mas IV baixo: são só 278 clientes. Testar com e sem (D5)"),
    "MonthlyIncome": (DESC, "Sem poder; base externa sem garantia de precisão (D12)"),
    "gender": (DESC, "Sem poder (D15)"), "PhoneService": (DESC, "Sem poder (D15)"),
    "MultipleLines": (DESC, "Sem poder e repete PhoneService (D15)"),
    "is_monthly_contract": (CAND, "Resume Contract em 1/0; escolher entre ela e Contract"),
    "no_support_with_internet": (CAND, "Internet sem suporte técnico"),
    "IsHighRiskContract": (CAND, "Mensal e cheque eletrônico: interação útil em modelo linear"),
    "HasFiberOpticWithoutTechSupport": (CAND, "Fibra sem suporte; versão restrita de no_support_with_internet"),
    "new_monthly_customer": (CAND, "Mensal e até 6 meses: interação útil em modelo linear"),
    "is_electronic_check": (CAND, "Guarda quase todo o IV de PaymentMethod em 1/0"),
    "is_new_customer": (CAND, "Até 6 meses; alternativa simples a tenure"),
    "pagamento_automatico": (CAND, "Oposto de pagamento manual; alternativa a is_electronic_check"),
    "n_protecao": (CAND, "Substitui total_addons e TotalServicesSubscribed, sem misturar streaming"),
    "n_streaming": (DESC, "Entre quem tem internet, streaming não muda o risco (ver os _bin)"),
    "mensalidade_relativa_pacote": (CAND, "Preço acima ou abaixo do pacote; alternativa a MonthlyCharges (D16)"),
    "charge_difference": (CAND, "Mudança de preço ao longo do tempo; escolher entre ela e ChargesRatio"),
    "ChargesRatio": (CAND, "Mesma ideia de charge_difference; escolher uma das duas"),
    "tenure_group": (DESC, "Para gráficos e apresentação; no modelo, tenure guarda mais informação"),
    "TenureCohort": (DESC, "Mesmo papel de tenure_group"),
    "tenure_years": (DESC, "É tenure ÷ 12: repete tenure"),
    "average_monthly_charges": (DESC, "Repete MonthlyCharges (Spearman 0,99)"),
    "total_addons": (DESC, "Mistura proteção e streaming; trocada por n_protecao e n_streaming"),
    "TotalServicesSubscribed": (DESC, "Mesmo problema de total_addons"),
    "IncomeCommitment": (DESC, "Divide pela renda, que não tem poder (D12)"),
    "income_after_charges": (DESC, "Quase a própria renda: sem poder"),
    "has_family": (DESC, "Não supera Dependents"),
}
# D10: o IV do _bin sozinho é baixo porque o "não tem" junta quem não contratou (≈40% de churn) com quem não tem
# internet (7%). Junto com InternetService, o que importa é o poder ENTRE QUEM TEM INTERNET, calculado aqui.
y_abt = abt_treino["Churn"]
com_internet = abt_treino["InternetService"].ne("No")
IV_INTERNET = {c: iv_woe(abt_treino.loc[com_internet, f"{c}_bin"], y_abt[com_internet]) for c in SERVICOS}
for c in SERVICOS:
    ivi = IV_INTERNET[c]
    churn = y_abt[com_internet].groupby(abt_treino.loc[com_internet, f"{c}_bin"]).mean() * 100
    txt = f"entre quem tem internet: IV {virg(ivi, 2)}, churn {virg(churn[0], 1)}% sem o serviço × {virg(churn[1], 1)}% com"
    if ivi >= 0.30:
        SUGESTAO[f"{c}_bin"] = (REC, f"Usar junto com InternetService (D10); {txt}")
        SUGESTAO[c] = (CAND, f"Versão com 3 categorias; alternativa: {c}_bin (D10)")
    elif ivi >= 0.10:
        SUGESTAO[f"{c}_bin"] = (CAND, f"Usar junto com InternetService, ou via n_protecao (D10); {txt}")
        SUGESTAO[c] = (CAND, f"Versão com 3 categorias; alternativa: {c}_bin ou n_protecao (D10)")
    else:
        SUGESTAO[f"{c}_bin"] = (DESC, f"Sem poder: {txt}")
        SUGESTAO[c] = (DESC, f"O IV de {virg(iv_orig[c], 2)} vinha só do No internet service; {txt}")

linhas = []
for ordem, col in enumerate(abt_treino.columns, start=1):
    if col in ("customerID", "Churn"):
        papel = "chave" if col == "customerID" else "alvo"
        linhas.append({"ordem": ordem, "coluna": col, "papel": papel, "origem": "original",
                       "descricao": meta.loc[col, "descricao"], "sugestao": papel, "motivo": "Não é variável do modelo" if papel == "chave" else "Variável resposta (1 = saiu)"})
        continue
    if col in meta.index:
        origem, tipo_d, formula = "original", "", ""
        descricao = meta.loc[col, "descricao"]
    elif col == "fl_total_charges_ausente":
        origem, tipo_d, formula, descricao = "indicador", "indicador de ausência (D5)", "1 se TotalCharges estava nulo", "Tinha TotalCharges ausente na base original"
    else:
        tipo_d, formula = ORIGEM_DERIV[col]
        origem, descricao = "derivada", formula
    iv = iv_coluna(abt_treino[col], y_abt)
    sug, motivo = SUGESTAO[col]
    linhas.append({"ordem": ordem, "coluna": col, "papel": "candidata", "origem": origem, "tipo_derivada": tipo_d, "formula": formula,
                   "descricao": descricao, "tipo": "número" if numerica(abt_treino[col]) else "texto",
                   "IV": round(iv, 3), "poder_iv": faixa_iv(iv),
                   "IV_entre_quem_tem_internet": round(IV_INTERNET[col[:-4]], 3) if col.endswith("_bin") else None,
                   "sugestao": sug, "motivo": motivo})
dicionario_abt = pd.DataFrame(linhas)
assert dicionario_abt["sugestao"].notna().all(), "coluna da ABT sem sugestão"
display(dicionario_abt.drop(columns=["descricao"]))
display(dicionario_abt.groupby("sugestao")["coluna"].agg(n_colunas="count", colunas=", ".join).reset_index())

In [ ]:
# 3.2 Gráfico: as colunas candidatas por IV, coloridas pela sugestão
cand = dicionario_abt[dicionario_abt["papel"] == "candidata"].sort_values("IV")
COR_SUG = {REC: COR, CAND: DESTAQUE, DESC: CINZA}
fig, ax = plt.subplots(figsize=(12, 13))
yy = np.arange(len(cand))
ax.barh(yy, cand["IV"], color=[COR_SUG[s] for s in cand["sugestao"]], height=0.65)
for y0, (_, r) in zip(yy, cand.iterrows()):
    ax.text(r["IV"] + 0.01, y0, f"{virg(r['IV'], 2)}", va="center", fontsize=8, color=TEXTO)
ib = cand["IV_entre_quem_tem_internet"]
ax.scatter(ib[ib.notna()], yy[ib.notna().values], marker="D", s=40, color=TEXTO, zorder=3,
           label="_bin: IV entre quem tem internet")
for y0, v in zip(yy[ib.notna().values], ib[ib.notna()]):
    if v >= 0.05:                                                   # perto de zero, o rótulo cobriria a barra
        ax.text(v + 0.012, y0 + 0.32, f"{virg(v, 2)} com internet", fontsize=7.5, color=TEXTO)
ax.set_yticks(yy, [f"{c} ({o})" if o != "original" else c for c, o in zip(cand["coluna"], cand["origem"])], fontsize=8)
for lim, rot in [(0.02, "sem poder | fraco"), (0.10, "fraco | médio"), (0.30, "médio | forte"), (0.50, "forte | muito forte")]:
    ax.axvline(lim, color="#cccccc", linestyle=":", linewidth=1, zorder=0)
    ax.text(lim, len(cand) + (0.5 if lim in (0.02, 0.30) else -0.3), rot, ha="left" if lim < 0.05 else "center", fontsize=8, color="#888888")
ax.set_ylim(-0.8, len(cand) + 0.6)
ax.set_xlabel("Information Value (IV), calculado na ABT de treino; losango = serviço _bin, só entre quem tem internet")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 1)))
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COR, label=f"recomendada ({(cand['sugestao'] == REC).sum()})"),
                   Patch(color=DESTAQUE, label=f"candidata: escolher entre alternativas ({(cand['sugestao'] == CAND).sum()})"),
                   Patch(color=CINZA, label=f"sugiro descartar ({(cand['sugestao'] == DESC).sum()})"),
                   plt.Line2D([], [], marker="D", color=TEXTO, linestyle="", label="serviço _bin: IV só entre quem tem internet")],
          loc="lower right", frameon=False, fontsize=9)
fig.suptitle(f"A ABT tem {len(cand)} colunas candidatas; a sugestão separa as recomendadas, as que têm alternativa\n"
             "na própria ABT e as que sugiro descartar",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "tenure", "InternetService", "MonthlyCharges", "TotalCharges"], fonte="gold_churn_treino (ABT de treino, montada a partir da silver_history)")
plt.show()

> **Como ler o dicionário e o gráfico 3.2**
> - **Cada linha é uma coluna da ABT**, com a origem (original, indicador ou derivada), o tipo de derivada e a fórmula.
> - **A cor é a sugestão:** azul = recomendada; laranja = candidata (há uma alternativa na ABT: escolher uma das duas);
>   cinza = sugiro descartar.
> - **O IV não é o único critério:** uma variável com IV alto pode repetir outra (por exemplo, `MonthlyCharges` e
>   `InternetService`). O motivo de cada sugestão está na coluna `motivo`.
> - **Para o modelo:** escolher **uma** de cada grupo de alternativas (por exemplo, `Contract` **ou**
>   `is_monthly_contract`; os serviços originais **ou** os `_bin`), para não repetir informação.
>
> **Dois achados que a ABT revelou**
> - **O IV dos serviços `_bin`, sozinho, é baixo** (0,02 a 0,16), porque o "não tem" junta quem não contratou o serviço
>   (cerca de 40% de churn) com quem não tem internet (7%). Junto com `InternetService`, o que conta é o poder **entre
>   quem tem internet** (coluna `IV_entre_quem_tem_internet`): segurança online (0,40) e suporte técnico (0,39) são
>   fortes; backup (0,17) e proteção de aparelho (0,13) são médios; **os streamings não têm poder** (0,00 e 0,01: churn de
>   33% sem e 31% com). O IV de 0,40 dos streamings no 03 vinha **só** do "No internet service". Por isso sugiro
>   descartar os streamings nas duas versões e o `n_streaming`.
> - **`fl_total_charges_ausente` tem IV de só 0,005**, apesar da diferença significativa da 4.3 (33,1% × 26,4%): são só
>   278 clientes, 5% da base. Fica como candidata, para testar com e sem (D5).

## 4. Gravar a camada gold

**O que faço:** gravo as quatro tabelas no catálogo e comento cada coluna com a descrição e a sugestão, para quem abrir a
tabela no Catalog Explorer saber o que é cada uma sem abrir este notebook.

In [ ]:
# 4.1 Tabela do texto (só para o modelo com texto, D9) e gravação da gold
TERMOS_CANCELAMENTO = (r"decided to churn|decided to cancel|recently canceled|recently cancelled|have canceled|switched to|"
                       r"decided to switch|decided to leave|had to churn|made the decision to churn|decision to churn")   # 7.3b
def tabela_texto(df: pd.DataFrame, base: str) -> pd.DataFrame:
    t = df[["customerID", "CustomerFeedback"]].copy()
    t["fl_menciona_cancelamento"] = t["CustomerFeedback"].str.lower().str.contains(TERMOS_CANCELAMENTO, regex=True).astype(int)
    t["base"] = base
    return t
texto = pd.concat([tabela_texto(silver_hist, "treino"), tabela_texto(silver_inf, "inferencia")], ignore_index=True)

def limpa(t: str) -> str:
    return str(t).replace("'", "").replace("\n", " ")[:250]

def gravar(df: pd.DataFrame, tabela: str, descricao: str, comentarios: dict):
    (spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{tabela}"))
    spark.sql(f"COMMENT ON TABLE {FQ}.{tabela} IS '{limpa(descricao)}'")
    for col, txt in comentarios.items():
        if col in df.columns:
            spark.sql(f"ALTER TABLE {FQ}.{tabela} ALTER COLUMN `{col}` COMMENT '{limpa(txt)}'")
    print(f"{FQ}.{tabela}: {virg(spark.table(f'{FQ}.{tabela}').count(), 0)} linhas · {df.shape[1]} colunas")

coment = {r["coluna"]: f"{r['descricao']} | {r['sugestao']}: {r['motivo']}" for _, r in dicionario_abt.iterrows()}
gravar(abt_treino, "gold_churn_treino", "ABT de treino: 5.634 clientes, todas as candidatas e Churn (1 = saiu). Ver gold_churn_dicionario", coment)
gravar(abt_inferencia, "gold_churn_inferencia", "ABT de inferência: 1.409 clientes, mesmas colunas do treino, sem Churn", coment)
gravar(dicionario_abt, "gold_churn_dicionario", "Uma linha por coluna da ABT: origem, tipo de derivada, fórmula, IV e sugestão da triagem (seleção no 05 e final na modelagem)", {})
gravar(texto, "gold_churn_texto", "SÓ para o modelo com texto (contraexemplo, D9): vazamento, não usar no modelo oficial",
       {"CustomerFeedback": "Comentário do cliente; escrito depois do desfecho (vazamento, 7.3 do 03)",
        "fl_menciona_cancelamento": "1 se o texto tem expressão de cancelamento (regra da 7.3b, acerta 98,9%)",
        "base": "treino ou inferencia"})

## Como usar a gold na modelagem

```python
FQ = "workspace.terra_signal"
treino = spark.table(f"{FQ}.gold_churn_treino").toPandas()
inferencia = spark.table(f"{FQ}.gold_churn_inferencia").toPandas()
dic = spark.table(f"{FQ}.gold_churn_dicionario").toPandas()

# modelo oficial (sem texto): escolher as colunas pelo dicionário
recomendadas = dic.loc[dic["sugestao"] == "recomendada", "coluna"].tolist()
X, y = treino[recomendadas], treino["Churn"]

# modelo com texto (contraexemplo, D9): juntar pelo customerID
texto = spark.table(f"{FQ}.gold_churn_texto").toPandas()
treino_txt = treino.merge(texto[texto["base"] == "treino"], on="customerID")
```

- **Categorias vêm como texto** (D14): cada modelo codifica do seu jeito (dummies na logística, direto em árvores).
- **O `customerID` nunca entra como variável**: serve para o `Prediction.csv`, com 1.409 linhas (D13).
- **Conferência da previsão final:** a taxa de churn real esperada na inferência é perto de 27%; a taxa de "Yes" do modelo fica perto de 40%, pelo balanceamento das classes e pelo limiar no máximo do F1 (notebook 08) (seção 5.9 do 03, sobre o PSI, *Population Stability Index*).

## Checklist final do 04

| # | Decisão | Status | Onde neste notebook |
|---|---|---|---|
| D14 | ABT com todas as candidatas, dicionário com sugestão, texto separado | ☑ Aplicado | Seções 2 a 4 |
| D18 | A preparação entrega a ABT e sugere; a seleção final é feita na modelagem | ☑ Aplicado | Coluna `sugestao` do dicionário |
| D19 | Seleção de variáveis | ☑ As 41 colunas do 05 (testes com correção FDR e retirada das duplicatas); a conferência com os métodos da aula cabe dentro delas (11/11) | Notebook 05 |
| D9 | Texto fora da ABT; `gold_churn_texto` para o modelo com texto | ☑ Aplicado | 2.2 (conferência) e 4.1 |
| D5 | `fl_total_charges_ausente` na ABT | ☑ Aplicado | Candidata, para testar com e sem |
| D10 | Serviços nas duas versões (original e `_bin`) | ◐ Sugestão: `_bin` com `InternetService`; segurança e suporte recomendadas, backup e proteção candidatas, streamings sem poder (sugiro descartar) | 2.1 e 3.1 |
| D12 | `MonthlyIncome` | ◐ Sugiro descartar; fica na ABT | 3.1 |
| D15 | `gender`, `PhoneService`, `MultipleLines` | ◐ Sugiro descartar; ficam na ABT | 3.1 |
| D16 | `MonthlyCharges` e `TotalCharges` | ◐ `MonthlyCharges` ou `mensalidade_relativa_pacote` candidatas; sugiro descartar `TotalCharges` | 3.1 |
| D13 | `Prediction.csv` com 1.409 linhas | ☑ Decidido | `gold_churn_inferencia` com 1.409 clientes |

**Saída do 04:** `gold_churn_treino`, `gold_churn_inferencia`, `gold_churn_dicionario` e `gold_churn_texto` em
`workspace.terra_signal`. **Próximo passo:** seleção de variáveis (05) e modelagem (06).